# 02 - Apache Iceberg com PySpark

Notebook com catálogo local e operações DML em Iceberg.

## 1. SparkSession com pacote Iceberg para Spark 3.5

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession

base = Path('..').resolve()
warehouse_dir = str((base / 'warehouse' / 'iceberg').resolve())

spark = (SparkSession.builder
 .appName('iceberg-demo')
 .master('local[*]')
 .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
 .config('spark.sql.catalog.local', 'org.apache.iceberg.spark.SparkCatalog')
 .config('spark.sql.catalog.local.type', 'hadoop')
 .config('spark.sql.catalog.local.warehouse', warehouse_dir)
 .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2')
 .getOrCreate())

print(warehouse_dir)

## 2. Carga da fonte CSV e criação de namespace/tabela

In [ ]:
df = spark.read.option('header', True).option('inferSchema', True).csv('../data/raw/vendas.csv')
df.createOrReplaceTempView('vendas_csv')

spark.sql('CREATE NAMESPACE IF NOT EXISTS local.demo')
spark.sql('DROP TABLE IF EXISTS local.demo.vendas')
spark.sql('''
CREATE TABLE local.demo.vendas (
  venda_id INT,
  cliente_id INT,
  produto_id INT,
  quantidade INT,
  desconto DOUBLE
) USING iceberg
''')

spark.sql('''
INSERT INTO local.demo.vendas
SELECT venda_id, cliente_id, produto_id, quantidade, desconto
FROM vendas_csv
''')

spark.sql('SELECT * FROM local.demo.vendas ORDER BY venda_id').show()

## 3. INSERT de evidência

In [ ]:
spark.sql('INSERT INTO local.demo.vendas VALUES (3001, 1, 102, 3, 0.0)')
spark.sql('SELECT * FROM local.demo.vendas WHERE venda_id = 3001').show()

## 4. UPDATE de evidência

In [ ]:
spark.sql('UPDATE local.demo.vendas SET desconto = 15.0 WHERE venda_id = 1003')
spark.sql('SELECT * FROM local.demo.vendas WHERE venda_id = 1003').show()

## 5. DELETE de evidência

In [ ]:
spark.sql('DELETE FROM local.demo.vendas WHERE venda_id = 1004')
spark.sql('SELECT * FROM local.demo.vendas ORDER BY venda_id').show()